In [17]:
import setup

setup.init_django()

In [18]:
from rag import (
    db as rag_db, 
    engines as rag_engines,
    settings as rag_settings, 
    updaters as rag_updaters,
)

In [19]:
from typing import Optional, Union
from sqlalchemy import create_engine, text

In [20]:
rag_settings.init()
rag_db.init_vector_db()
rag_updaters.update_llama_index_documents(use_saved_embeddings=True)

OperationalError: terminating connection due to administrator command

In [ ]:
vector_index = rag_engines.get_semantic_query_index()
semantic_query_retriever = rag_engines.get_semantic_query_retriever_engine()
sql_query_engine = rag_engines.get_sql_query_engine()

In [ ]:
print(rag_settings.VECTOR_DB_NAME, rag_settings.VECTOR_DB_TABLE_NAME)

In [ ]:
from llama_index.core.tools import QueryEngineTool

vector_tool = QueryEngineTool.from_defaults(
    query_engine=semantic_query_retriever,
    description=(
        f"Useful for answering semantic questions about different blog posts"
    ),
)

In [ ]:
sql_tool = QueryEngineTool.from_defaults(
    query_engine=sql_query_engine,
    description=(
        "Useful for translating a natural language query into a SQL query over"
        " a table containing: blog posts and page views each blog post"
    ),
)

In [ ]:
from typing import Any, Optional, Union


from llama_index.core.query_engine import SQLAutoVectorQueryEngine
from llama_index.core.query_engine.sql_vector_query_engine import *
from llama_index.core.service_context import ServiceContext


class MySQLAutoVectorQueryEngine(SQLAutoVectorQueryEngine):
    def __init__(
        self,
        sql_query_tool: QueryEngineTool,
        vector_query_tool: QueryEngineTool,
        selector: Optional[Union[LLMSingleSelector, PydanticSingleSelector]] = None,
        llm: Optional[LLM] = None,
        service_context: Optional[ServiceContext] = None,
        sql_vector_synthesis_prompt: Optional[BasePromptTemplate] = None,
        sql_augment_query_transform: Optional[SQLAugmentQueryTransform] = None,
        use_sql_vector_synthesis: bool = True,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = True,
    ) -> None:
        """Initialize params."""
        # validate that the query engines are of the right type
        if not isinstance(
            sql_query_tool.query_engine,
            (BaseSQLTableQueryEngine, NLSQLTableQueryEngine),
        ):
            raise ValueError(
                "sql_query_tool.query_engine must be an instance of "
                "BaseSQLTableQueryEngine or NLSQLTableQueryEngine"
            )
        if not isinstance(vector_query_tool.query_engine, RetrieverQueryEngine):
            raise ValueError(
                "vector_query_tool.query_engine must be an instance of "
                "RetrieverQueryEngine"
            )
        # if not isinstance(
        #     vector_query_tool.query_engine.retriever, VectorIndexAutoRetriever
        # ):
        #     raise ValueError(
        #         "vector_query_tool.query_engine.retriever must be an instance "
        #         "of VectorIndexAutoRetriever"
        #     )

        sql_vector_synthesis_prompt = (
            sql_vector_synthesis_prompt or DEFAULT_SQL_VECTOR_SYNTHESIS_PROMPT
        )
        SQLJoinQueryEngine.__init__(
            self,
            sql_query_tool,
            vector_query_tool,
            selector=selector,
            llm=llm,
            
            sql_join_synthesis_prompt=sql_vector_synthesis_prompt,
            sql_augment_query_transform=sql_augment_query_transform,
            use_sql_join_synthesis=use_sql_vector_synthesis,
            callback_manager=callback_manager,
            verbose=verbose,
        )

In [ ]:
# from llama_index.core.query_engine import SQLAutoVectorQueryEngine

query_engine = MySQLAutoVectorQueryEngine(
    sql_tool, 
    vector_tool,
)

In [ ]:
response = query_engine.query(
    "What kind of org is discussed?"
)

In [ ]:
response.response

In [21]:
response = query_engine.query(
    "Are are the top 5 most viewed blog posts? What keywords do their content have?"
)

Querying SQL database: The question requires translating a natural language query into a SQL query to find the top 5 most viewed blog posts, which aligns with choice 1 as it involves querying a table with blog posts and page views.
SQL query: SELECT bp.id, bp.title, bp.content, COUNT(ap.id) AS view_count
FROM blog_blogpost bp
JOIN analytics_pageview ap ON bp.id = ap.post_id
GROUP BY bp.id, bp.title, bp.content
ORDER BY view_count DESC
LIMIT 5;
SQL response: The top 5 most viewed blog posts, based on the query results, are as follows:

1. **Blog Post 2**: "The cat jumped over the dog" with 2,366 views.
   - **Keywords**: cat, dog, jumped

2. **Blog Post 1**: "The dog jumped over the cat" with 2,139 views.
   - **Keywords**: dog, cat, jumped

3. **Blog Post 3**: "The weather is very hot" with 1,553 views.
   - **Keywords**: weather, hot

4. **Blog Post 4**: "The cat is yellow and dog is red" with 654 views.
   - **Keywords**: cat, yellow, dog, red

5. **The lifelong fan**: "What does it 

OperationalError: (psycopg2.OperationalError) SSL SYSCALL error: EOF detected

[SQL: SELECT public.data_blogpost.id, public.data_blogpost.node_id, public.data_blogpost.text, public.data_blogpost.metadata_, public.data_blogpost.embedding <=> %(embedding_1)s AS distance 
FROM public.data_blogpost ORDER BY distance asc 
 LIMIT %(param_1)s]
[parameters: {'embedding_1': '[0.0011830402072519064,0.02388329803943634,-0.03365977481007576,0.0050210715271532536,0.01979205384850502,-0.03740565478801727,-0.05127337574958801,0 ... (32357 characters truncated) ... 6,0.001188851660117507,0.0198186207562685,-0.01951310597360134,-0.016803322359919548,-0.018012098968029022,0.038282349705696106,-0.02287377044558525]', 'param_1': 5}]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [22]:
from IPython.display import Markdown, display

display(Markdown(response.response))

The discussion contrasts two types of entities: organizations and organisms. An organization is described as structured and controlled, with systems and charts that require approval for changes. In contrast, an organism is characterized by constant change, adaptation, and resilience, similar to a living culture that thrives by understanding the system it inhabits.

In [23]:
response = query_engine.query(
    "What are the top 5 least viewed blog posts from today?"
)
print(response.response)

Querying SQL database: The question requires translating a natural language query into a SQL query to find the top 5 least viewed blog posts from today, which aligns with choice 1's description of translating queries over a table containing blog posts and page views.
SQL query: SELECT blog_blogpost.id, blog_blogpost.title, COUNT(analytics_pageview.id) AS view_count
FROM blog_blogpost
LEFT JOIN analytics_pageview ON blog_blogpost.id = analytics_pageview.post_id
WHERE analytics_pageview.timestamp::date = CURRENT_DATE
GROUP BY blog_blogpost.id, blog_blogpost.title
ORDER BY view_count ASC
LIMIT 5;
SQL response: It looks like there are no blog posts that have been viewed today, or there might be an issue with the data collection for today's views. As a result, there are no entries to display for the least viewed blog posts. If you believe this is an error, you might want to check the data collection process or ensure that there are blog posts published and accessible today.
Transformed quer

In [24]:
display(Markdown(response.response))

It looks like there are no blog posts that have been viewed today, or there might be an issue with the data collection for today's views. As a result, there are no entries to display for the least viewed blog posts. If you believe this is an error, you might want to check the data collection process or ensure that there are blog posts published and accessible today.